In [1]:
import joblib
import requests
from fastapi import FastAPI
from pydantic import BaseModel # let you make a structure of input with class
from fastapi.testclient import TestClient
import pandas as pd

In [2]:
app=FastAPI(title="ml_defects")
loaded_artifact = joblib.load("Machine defect_model.joblib")# model
#############

model=loaded_artifact["pipeline"] #dt model
key=loaded_artifact.keys()
threshold=loaded_artifact["threshold"]#0.5
features=loaded_artifact["features"]

##########

In [3]:
class SensorData (BaseModel):
    Temperature:float
    Pressure:float
    Vibration_Level:float
    Humidity:float
    Power_Consumption:float



data = SensorData(
    Temperature=75.5,
    Pressure=0.9,
    Vibration_Level=0.8,
    Humidity=0.80,
    Power_Consumption=0.125
)


In [4]:
@app.get("/")
def get():
    return{"message":"API is running...."}

@app.post("/predict")
def predict(data:SensorData):
    input_data1 = pd.DataFrame([data.model_dump()])[features]
    prob=float(model.predict_proba(input_data1)[0,1])
    pred=int(prob>=threshold)
    return{
        "failure_probability":prob,
        "prediction":pred,
        "label":"Failure" if pred==1 else "No_failure"
    }

In [5]:
client1=TestClient(app)
response1=client1.get("/")
print(response1.status_code)
print(response1.json())

200
{'message': 'API is running....'}


In [6]:
client2 = TestClient(app)

response2 = client2.post("/predict",json=data.model_dump())

print(response2.status_code)
print(response2.json())

200
{'failure_probability': 0.0, 'prediction': 0, 'label': 'No_failure'}


In [7]:
print([route.path for route in app.routes])

['/openapi.json', '/docs', '/docs/oauth2-redirect', '/redoc', '/', '/predict']
